<a href="https://colab.research.google.com/github/Alex-Devoid/ST554-HW/blob/main/HW6_Alex_Devoid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW6 — SQL Practice and Classes

Name: Alex Devoid  
Course: ST 554

## Part I — More Practice Querying a Database

In [8]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1) Look at all of the tables in the database
Connecting to the Lahman SQLite database and looking at all of the tables in the database.
`pd.read_sql()` willl return the results as a pandas data frame.


In [9]:


DB_FILE = "/content/lahman_1871-2022.sqlite"

# sqlite3.connect() opens the database file so SQL queries can be sent to it
conn = sqlite3.connect(DB_FILE)

In [10]:
# Look at all tables in the database
# sqlite_schema is SQLite's catalog table, so querying it lets us list the tables in the database
tables_df = pd.read_sql("""
SELECT name
FROM sqlite_schema
WHERE type = 'table';
""", conn)

tables_df

,name
0,AllstarFull
1,Appearances
2,AwardsManagers
3,AwardsPlayers
4,AwardsShareManagers
5,AwardsSharePlayers
6,Batting
7,BattingPost
8,CollegePlaying
9,Fielding


### 2.

Hall of Fame pitchers

For this part, I want Hall of Famers who pitched and their total values for:
- `GS`
- `G`
- `W`
- `L`
- `IPOuts`
- `CG`
- `SHO`
- `SV`

I am including regular season and postseason pitching records.

In [11]:
# Hall of Fame pitchers and pitching rows
# useing SQL first to filter down to the right players and rows,
# then useing pandas to total the stats into one row per pitcher

hof_pitching_raw = pd.read_sql("""
SELECT
    -- SELECT is used to choose only the columns we actually need
    hp.playerID,
    p.GS,
    p.G,
    p.W,
    p.L,
    p.IPOuts,
    p.CG,
    p.SHO,
    p.SV
FROM (
    SELECT DISTINCT h.playerID
    -- DISTINCT is used so each Hall of Fame pitcher appears only once here
    FROM HallOfFame AS h
    INNER JOIN (
        SELECT DISTINCT playerID
        FROM Pitching
        UNION
        SELECT DISTINCT playerID
        FROM PitchingPost
        -- UNION is used here to combine pitcher IDs from both tables and remove duplicates
    ) AS p_ids
        ON h.playerID = p_ids.playerID
        -- INNER JOIN is used to keep only players who are in both the HallOfFame table and the pitching ID list
    WHERE h.inducted = 'Y'
    -- WHERE is used to filter to players who were actually inducted into the Hall of Fame
) AS hp
LEFT JOIN (
    SELECT playerID, GS, G, W, L, IPOuts, CG, SHO, SV
    FROM Pitching
    UNION ALL
    SELECT playerID, GS, G, W, L, IPOuts, CG, SHO, SV
    FROM PitchingPost
    -- UNION ALL is used here instead of UNION because we want to keep every stat row from both regular season and postseason so we can total all of them later
) AS p
    ON hp.playerID = p.playerID
    -- LEFT JOIN is used so every Hall of Fame pitcher stays in the result
;
""", conn)

# these are the pitching columns we want to total for each pitcher
pitch_cols = ["GS", "G", "W", "L", "IPOuts", "CG", "SHO", "SV"]

hof_pitching = (
    hof_pitching_raw

    .fillna({col: 0 for col in pitch_cols})
    # groupby is used to collapse multiple rows per pitcher into one row

    .groupby("playerID", as_index=False)[pitch_cols]
    .sum()
)

hof_pitching.head(), hof_pitching.shape

(    playerID   GS    G    W    L  IPOuts   CG  SHO  SV
 0  alexape01  604  703  376  210   15699  441   90  33
 1  ansonca01    0    3    0    1      12    0    0   1
 2  becklja01    1    1    0    1      12    0    0   0
 3  bendech01  344  469  218  131    9306  264   41  34
 4  blylebe01  691  700  292  251   15052  243   60   0,
 (108, 9))